# Notebook 04 - Train the Consensus MLP (23-dim numerical features)

This notebook trains the **consensus fusion head**. After Agent A (EfficientNet-B4) and Agent B (ViT-B/16) classify a lesion and the debate trigger fires, the consensus MLP fuses the two agents' signals into a single calibrated 8-class prediction.

The previous 788-dim Groq-LLM-debate-text + sentence-embedding contract has been **removed entirely** and replaced with a fixed **23-dimensional pure numerical** feature vector (identical to `backend/ml/debate/features.py` and `05_evaluation.ipynb`):

| Component | Dim | Meaning |
|-----------|-----|---------|
| `pA` | 8 | Agent A softmax probabilities |
| `pB` | 8 | Agent B softmax probabilities |
| disagreement | 4 | JS divergence, entropy_A, entropy_B, max prob delta |
| attention | 3 | attn IoU, attn entropy_A, attn entropy_B (0.0 if unavailable) |

Reasons for the change: Groq rate-limited after 2-3 images (unusable for thousands of calls); ~100% of previous arguments were deterministic fallback templates, not real LLM output; 788 features with ~1,500 samples is a statistically indefensible 1.9:1 ratio, whereas 23 features give ~65:1. The architecture and parameter names match `backend/ml/consensus/classifier.py` so the checkpoint loads in the app.

## Exact Kaggle setup (do this before Run All)
1. **Attach the data**: Add Data -> search **`andrewmvd/isic-2019`** -> Add.
2. **Attach the previous checkpoints**: the outputs of Notebook 01 (`agent_a_best.pth`) and Notebook 02 (`agent_b_best.pth`). `find_file()` locates them under `/kaggle/input`.
3. **Accelerator**: Settings -> Accelerator -> **GPU T4 x2** (or **P100**).
4. **Internet**: Settings -> Internet -> **ON** (pip + pretrained timm weights). No Groq / sentence-transformers / API key is needed any more.
5. **Run All**.

> Outputs: `consensus_best.pth`, `consensus_scaler.pkl` (+ `.json` sidecar) and `consensus_temperature.txt`.

## 1. Install extras, imports, shared helpers and device

In [ ]:
# Kaggle pre-installs torch, torchvision, numpy, pandas, scikit-learn, matplotlib,
# seaborn, opencv-python, Pillow, tqdm. We install the EXTRAS this notebook needs.
# INTERNET MUST BE ON (Settings -> Internet -> ON) for pip + pretrained weights.
import sys, subprocess

print("Installing extras (timm, grad-cam, netcal)...")
print("NOTE: Internet must be ON for this to work (Settings -> Internet -> ON). No Groq / sentence-transformers / API key is used.")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-U", "timm",                 # -U so the ViT model id resolves
    "lightgbm",                   # LightGBM consensus head (replaces MLP)
    "grad-cam",                   # Grad-CAM++ for Agent A
    "netcal",                     # ECE metric
], check=False)

import os, json, math, time, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import joblib
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# --- Shared contract constants (identical to backend + other notebooks) ----
ISIC_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
NUM_CLASSES = 8

# Consensus feature contract (must match backend/ml/debate/features.py and 05_evaluation.ipynb).
FEATURE_DIM = 23

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Reproducibility.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def discover_isic(root="/kaggle/input"):
    """Locate the ISIC-2019 ground-truth CSV and the training-image directory.

    The CSV is the .csv whose header contains ALL 8 ISIC class names. The image
    dir is the directory containing the most ISIC_*.jpg/.jpeg files. Robust to
    nested mirror folders (e.g. doubled ISIC_2019_Training_Input/).
    """
    csv_path = None
    class_set = set(ISIC_CLASSES)
    for dirpath, _dirnames, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith(".csv"):
                fp = os.path.join(dirpath, fn)
                try:
                    cols = set(pd.read_csv(fp, nrows=0).columns)
                except Exception:
                    continue
                if class_set.issubset(cols):
                    csv_path = fp
                    break
        if csv_path is not None:
            break

    # Image dir = directory with the most ISIC_*.jpg/.jpeg files.
    best_dir = None
    best_count = 0
    for dirpath, _dirnames, filenames in os.walk(root):
        count = 0
        for fn in filenames:
            low = fn.lower()
            if low.startswith("isic_") and (low.endswith(".jpg") or low.endswith(".jpeg")):
                count += 1
        if count > best_count:
            best_count = count
            best_dir = dirpath

    print("Discovered ground-truth CSV :", csv_path)
    print("Discovered image directory  :", best_dir, "(%d ISIC images)" % best_count)
    assert csv_path is not None and os.path.exists(csv_path), "Could not find ISIC ground-truth CSV under " + root
    assert best_dir is not None and best_count > 0, "Could not find an ISIC image directory under " + root
    return csv_path, best_dir


def find_file(filename_substring, search_roots=("/kaggle/input", "/kaggle/working")):
    """Return the first path whose basename contains the given substring."""
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                if filename_substring in fn:
                    found = os.path.join(dirpath, fn)
                    print("find_file('%s') -> %s" % (filename_substring, found))
                    return found
    print("find_file('%s'): NOT FOUND under %s" % (filename_substring, list(search_roots)))
    return None

## 3. Load Agent A and Agent B

Rebuild the two agents exactly as in Notebook 03: EfficientNet-B4 (Agent A) and ViT-B/16 (Agent B) via `timm`, loading their trained checkpoints with `find_file`. The sentence encoder has been removed (no more text embeddings).

In [ ]:
import timm

AGENT_A_MODEL = "efficientnet_b4"
AGENT_B_MODEL = "vit_base_patch16_224.augreg_in21k_ft_in1k"


def load_agent(model_name, ckpt_substring):
    """Create a timm backbone with an 8-class head and load its checkpoint."""
    ckpt_path = find_file(ckpt_substring)
    if ckpt_path is not None and os.path.exists(ckpt_path):
        model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
        state = torch.load(ckpt_path, map_location=DEVICE)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        res = model.load_state_dict(state, strict=False)
        print("Loaded %s from %s (missing=%d, unexpected=%d)" % (
            model_name, ckpt_path, len(res.missing_keys), len(res.unexpected_keys)))
    else:
        print("WARNING: no checkpoint for %s; using ImageNet-pretrained backbone "
              "with a random 8-class head (predictions NOT clinically meaningful)." % model_name)
        model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    model.eval().to(DEVICE)
    return model


agent_a = load_agent(AGENT_A_MODEL, "agent_a_best")
agent_b = load_agent(AGENT_B_MODEL, "agent_b_best")

## 4. Dataset, the SAME stratified train split, and the debate trigger

We build the ISIC-2019 dataset from the one-hot ground-truth CSV (label = argmax over the 8
class columns), then create the **same** stratified train/val split used everywhere
(`test_size=0.15, stratify=labels, random_state=42`). We then run both agents over the TRAIN
images and select those where the **debate trigger fires** (squared Jensen-Shannon divergence
> tau, or either agent's Shannon entropy > tau). Optionally we restrict to Notebook 03's
`hard_subset.csv` when it is attached.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

BATCH_SIZE = 32  # lower to 16 if you hit CUDA OOM
NUM_WORKERS = 2

CSV_PATH, IMAGE_DIR = discover_isic()

df = pd.read_csv(CSV_PATH)
# image-id column is named "image" in the ISIC-2019 ground truth.
id_col = "image" if "image" in df.columns else df.columns[0]
labels_all = df[ISIC_CLASSES].values.argmax(axis=1).astype(np.int64)
df = df.assign(_label=labels_all)
print("Total labelled rows:", len(df))
print("Class counts:", {ISIC_CLASSES[i]: int((labels_all == i).sum()) for i in range(NUM_CLASSES)})

# The SAME stratified split used across the project.
train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["_label"].values, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("Train rows:", len(train_df), "Val rows:", len(val_df))

# Optionally restrict to Notebook 03's hard subset, if attached.
# NOTE: NB03 mines D_hard from the held-out (val) split, so we match
# hard_ids against the FULL ground-truth `df`, not just `train_df`.
# The consensus MLP's own 80/20 internal split provides its holdout.
hard_csv = find_file("hard_subset.csv")
if hard_csv is not None:
    try:
        hard_df = pd.read_csv(hard_csv)
        if "image_path" in hard_df.columns:
            hard_ids = set(
                hard_df["image_path"].astype(str).apply(
                    lambda p: os.path.splitext(os.path.basename(p))[0]
                ).tolist()
            )
        elif "image" in hard_df.columns:
            hard_ids = set(hard_df["image"].astype(str).tolist())
        else:
            hard_ids = set(hard_df[hard_df.columns[0]].astype(str).tolist())
        before = len(train_df)
        train_df = df[df[id_col].astype(str).isin(hard_ids)].reset_index(drop=True)
        print("Restricted to hard_subset.csv (matched against full dataset): %d -> %d rows" % (before, len(train_df)))
    except Exception as exc:
        print("Could not apply hard_subset.csv (%s); using full train split." % exc)


# val-style transform (deterministic) for trigger scan + feature building.
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# train-style augmentation transform (documented for completeness; the consensus
# head trains on extracted features, so feature extraction uses eval_tf).
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])


class ISICDataset(Dataset):
    """Reads the one-hot ISIC-2019 ground-truth CSV; label = argmax over 8 classes."""

    def __init__(self, frame, image_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image_id = str(row[id_col])
        path = os.path.join(self.image_dir, image_id + ".jpg")
        img = Image.open(path).convert("RGB")
        tensor = self.transform(img)
        label = int(row["_label"])
        return tensor, label, path


# --- Trigger (inlined from backend/ml/debate/trigger.py) -------------------
from scipy.spatial.distance import jensenshannon

TAU_JS = 0.25        # squared Jensen-Shannon divergence threshold
TAU_ENTROPY = 0.8   # Shannon entropy (bits) threshold
_EPS = 1e-12


def shannon_entropy(probs):
    p = np.clip(np.asarray(probs, dtype=np.float64), 0.0, None)
    return -float(np.sum(p * np.log2(p + _EPS)))


def trigger_fired(pa, pb, tau_js=TAU_JS, tau_entropy=TAU_ENTROPY):
    pa = np.asarray(pa, dtype=np.float64).ravel()
    pb = np.asarray(pb, dtype=np.float64).ravel()
    jsd = jensenshannon(pa, pb, base=2)
    jsd = float(jsd) ** 2
    if not np.isfinite(jsd):
        jsd = 0.0
    ea = shannon_entropy(pa)
    eb = shannon_entropy(pb)
    fired = (jsd > tau_js) or (max(ea, eb) > tau_entropy)
    return fired, jsd, ea, eb


@torch.no_grad()
def agent_probs(model, tensor):
    """Return the 8-class softmax probability vector (numpy) for a (1,3,224,224) tensor."""
    logits = model(tensor.to(DEVICE))
    return F.softmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.float64)


# Scan the TRAIN pool and keep images where the trigger fires.
# Cap the number of fired images so a Kaggle session finishes comfortably; raise
# MAX_FIRED if you have time/credits to spare.
MAX_SCAN = 8000     # how many train rows to scan for triggers
MAX_FIRED = 5000     # raised: hard_subset.csv has ~3675 rows; cap must exceed that

scan_df = train_df.iloc[:min(MAX_SCAN, len(train_df))].reset_index(drop=True)
scan_ds = ISICDataset(scan_df, IMAGE_DIR, eval_tf)

fired_records = []  # each: dict(image_id, path, label, pa, pb, jsd)
print("Scanning %d train images for trigger firing..." % len(scan_ds))
for i in tqdm(range(len(scan_ds))):
    tensor, label, path = scan_ds[i]
    tensor = tensor.unsqueeze(0)
    pa = agent_probs(agent_a, tensor)
    pb = agent_probs(agent_b, tensor)
    fired, jsd, ea, eb = trigger_fired(pa, pb)
    if fired:
        fired_records.append({
            "image_id": os.path.splitext(os.path.basename(path))[0],
            "path": path,
            "label": int(label),
            "pa": pa,
            "pb": pb,
            "jsd": float(jsd),
        })
    if len(fired_records) >= MAX_FIRED:
        break

print("Trigger fired on %d images (building consensus features for these)." % len(fired_records))
assert len(fired_records) > 0, "No images fired the trigger; lower TAU_JS / TAU_ENTROPY or raise MAX_SCAN."

## 5. Build the 23-d consensus features for every fired image

For each fired image we compute **Grad-CAM++** (Agent A) and **attention rollout** (Agent B) saliency maps, then assemble the 23-dim vector via `extract_consensus_features` (pA, pB, JS divergence, entropies, max prob delta, attention IoU/entropies). No LLM debate, no sentence embeddings. Per-feature diagnostics are printed to catch degenerate columns.

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# --- Grad-CAM++ for Agent A (target layer resolution mirrors the backend) ---
def _resolve_target_layer(model):
    blocks = getattr(model, "blocks", None)
    if blocks is not None and len(blocks) > 0:
        last_block = blocks[-1]
        bn3 = getattr(last_block, "bn3", None)
        if isinstance(bn3, nn.Module):
            return bn3
        return last_block
    conv_head = getattr(model, "conv_head", None)
    if isinstance(conv_head, nn.Module):
        return conv_head
    fallback = None
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm2d, nn.Conv2d)):
            fallback = module
    if fallback is None:
        raise RuntimeError("No Grad-CAM++ target layer found.")
    return fallback


def compute_gradcam_plusplus(model, tensor, target_class):
    target_layer = _resolve_target_layer(model)
    model.zero_grad()
    cam = GradCAMPlusPlus(model=model, target_layers=[target_layer])
    grayscale = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(int(target_class))])
    model.zero_grad()
    heat = np.asarray(grayscale[0], dtype=np.float32)
    if heat.shape != (IMAGE_SIZE, IMAGE_SIZE):
        heat = cv2.resize(heat, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
    return np.ascontiguousarray(np.clip(heat, 0.0, 1.0), dtype=np.float32)


# --- Attention rollout for Agent B (inlined from backend/ml/attention/rollout.py) ---
GRID_SIZE = 14
NUM_PATCH_TOKENS = GRID_SIZE * GRID_SIZE


def _centered_gaussian(size=IMAGE_SIZE, sigma_frac=0.25):
    coords = np.linspace(-1.0, 1.0, size, dtype=np.float32)
    gx, gy = np.meshgrid(coords, coords)
    sigma = max(sigma_frac, 1e-6)
    g = np.exp(-(gx ** 2 + gy ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    span = float(g.max() - g.min())
    if span < 1e-12:
        return np.zeros((size, size), dtype=np.float32)
    return ((g - g.min()) / span).astype(np.float32)


def compute_attention_rollout(model, tensor):
    handles = []
    captured = {}
    original_fused = {}
    blocks = getattr(model, "blocks", None)
    if blocks is None or len(blocks) == 0:
        return _centered_gaussian()

    def _make_hook(layer_idx):
        def _hook(module, inputs, output):
            try:
                x = inputs[0]
                batch, num_tokens, dim = x.shape
                num_heads = int(module.num_heads)
                head_dim = dim // num_heads
                scale = getattr(module, "scale", None)
                if scale is None:
                    scale = head_dim ** -0.5
                qkv = module.qkv(x).reshape(batch, num_tokens, 3, num_heads, head_dim)
                qkv = qkv.permute(2, 0, 3, 1, 4)
                q, k = qkv[0], qkv[1]
                attn = (q @ k.transpose(-2, -1)) * float(scale)
                attn = attn.softmax(dim=-1)
                captured[layer_idx] = attn.mean(dim=1).detach().to(torch.float32).cpu()
            except Exception:
                pass
        return _hook

    try:
        for idx, block in enumerate(blocks):
            attn_module = getattr(block, "attn", None)
            if attn_module is None or not hasattr(attn_module, "qkv"):
                continue
            original_fused[idx] = bool(getattr(attn_module, "fused_attn", False))
            if hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = False
            handles.append(attn_module.register_forward_hook(_make_hook(idx)))
        if not handles:
            return _centered_gaussian()
        with torch.no_grad():
            model(tensor)
        if not captured:
            return _centered_gaussian()
        layer_indices = sorted(captured.keys())
        num_tokens = captured[layer_indices[0]].shape[-1]
        identity = torch.eye(num_tokens, dtype=torch.float32)
        rollout = torch.eye(num_tokens, dtype=torch.float32)
        for idx in layer_indices:
            attn = captured[idx][0]
            aug = 0.5 * attn + 0.5 * identity
            aug = aug / aug.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            rollout = aug @ rollout
        cls_attention = rollout[0, 1:]
        if cls_attention.shape[0] < NUM_PATCH_TOKENS:
            return _centered_gaussian()
        grid = cls_attention[:NUM_PATCH_TOKENS].reshape(GRID_SIZE, GRID_SIZE).numpy().astype(np.float32)
        heat = cv2.resize(grid, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_CUBIC).astype(np.float32)
        span = float(heat.max() - heat.min())
        if span < 1e-12:
            return _centered_gaussian()
        heat = (heat - heat.min()) / span
        return np.ascontiguousarray(np.clip(heat, 0.0, 1.0), dtype=np.float32)
    except Exception:
        return _centered_gaussian()
    finally:
        for handle in handles:
            handle.remove()
        for idx, was_fused in original_fused.items():
            attn_module = getattr(blocks[idx], "attn", None)
            if attn_module is not None and hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = was_fused

print("Attention helpers ready (Grad-CAM++ for Agent A, rollout for Agent B).")


In [ ]:
# --- Canonical 23-dim consensus feature extractor -------------------------
# IDENTICAL to ml_training/debate_text_utils.py, backend/ml/debate/features.py and
# 05_evaluation.ipynb. If you change this you MUST change it in all four places and
# retrain the consensus MLP, else train/eval/serve features silently disagree.
from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import jensenshannon

CLASS_NAMES = ISIC_CLASSES

FEATURE_NAMES = (
    [f"pA_{c}" for c in CLASS_NAMES]
    + [f"pB_{c}" for c in CLASS_NAMES]
    + ["js_div", "entropy_a", "entropy_b", "max_prob_delta",
       "attn_iou", "attn_entropy_a", "attn_entropy_b"]
)


def extract_consensus_features(prob_a, prob_b, attn_map_a=None, attn_map_b=None):
    """Return the 23-d consensus feature vector for one image.

    Layout: [pA(8), pB(8), js_div, entropy_a, entropy_b, max_prob_delta,
             attn_iou, attn_entropy_a, attn_entropy_b]. The three attention
    features are 0.0 when either attention map is None.
    """
    prob_a = np.asarray(prob_a, dtype=np.float64).flatten()
    prob_b = np.asarray(prob_b, dtype=np.float64).flatten()

    prob_a = np.clip(prob_a, 1e-9, 1.0)
    prob_a /= prob_a.sum()
    prob_b = np.clip(prob_b, 1e-9, 1.0)
    prob_b /= prob_b.sum()

    js_div = float(jensenshannon(prob_a, prob_b) ** 2)
    entropy_a = float(scipy_entropy(prob_a, base=2))
    entropy_b = float(scipy_entropy(prob_b, base=2))
    max_prob_delta = float(np.max(np.abs(prob_a - prob_b)))

    if attn_map_a is not None and attn_map_b is not None:
        a = np.asarray(attn_map_a, dtype=np.float32)
        b = np.asarray(attn_map_b, dtype=np.float32)
        a = (a - a.min()) / (a.max() - a.min() + 1e-9)
        b = (b - b.min()) / (b.max() - b.min() + 1e-9)
        mask_a = (a >= 0.5).astype(np.float32)
        mask_b = (b >= 0.5).astype(np.float32)
        intersection = (mask_a * mask_b).sum()
        union = np.clip(mask_a + mask_b, 0, 1).sum()
        attn_iou = float(intersection / (union + 1e-9))
        a_flat = a.flatten() + 1e-9
        a_flat /= a_flat.sum()
        b_flat = b.flatten() + 1e-9
        b_flat /= b_flat.sum()
        attn_entropy_a = float(scipy_entropy(a_flat, base=2))
        attn_entropy_b = float(scipy_entropy(b_flat, base=2))
    else:
        attn_iou = 0.0
        attn_entropy_a = 0.0
        attn_entropy_b = 0.0

    feat = np.concatenate([
        prob_a,
        prob_b,
        [js_div, entropy_a, entropy_b, max_prob_delta,
         attn_iou, attn_entropy_a, attn_entropy_b],
    ]).astype(np.float32)

    assert feat.shape == (FEATURE_DIM,), \
        f"Feature dim mismatch: got {feat.shape}, expected ({FEATURE_DIM},)"
    assert not np.any(np.isnan(feat)), "NaN in feature vector"
    assert not np.any(np.isinf(feat)), "Inf in feature vector"
    return feat


print("extract_consensus_features ready (FEATURE_DIM=%d)." % FEATURE_DIM)


In [ ]:
# --- Build the 23-d feature matrix + labels over all fired images ----------
print("Building 23-dim consensus features for %d fired images..." % len(fired_records))

feat_eval_ds = ISICDataset(
    pd.DataFrame({id_col: [r["image_id"] for r in fired_records],
                  "_label": [r["label"] for r in fired_records]}),
    IMAGE_DIR, eval_tf)

X_list, y_list = [], []
attn_fail = 0
for i, rec in enumerate(tqdm(fired_records)):
    tensor, label, _path = feat_eval_ds[i]
    tensor = tensor.unsqueeze(0).to(DEVICE)
    pa = rec["pa"]; pb = rec["pb"]
    try:
        cam_input = tensor.clone().requires_grad_(True)
        heat_a = compute_gradcam_plusplus(agent_a, cam_input, int(np.argmax(pa)))
        heat_b = compute_attention_rollout(agent_b, tensor)
    except Exception as exc:
        attn_fail += 1
        print("  attention failed for %s (%s); attention features set to 0.0" % (rec["image_id"], exc))
        heat_a, heat_b = None, None
    feat = extract_consensus_features(pa, pb, heat_a, heat_b)
    X_list.append(feat)
    y_list.append(int(label))

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.int64)
if attn_fail:
    print("WARNING: attention maps failed for %d image(s); their 3 attention features are 0.0." % attn_fail)

# --- Verify the feature matrix (shape, NaN/Inf, degenerate columns) --------
print("Feature matrix:", X.shape, " labels:", y.shape)
assert X.shape[1] == 23, "Expected 23 features, got %d" % X.shape[1]
assert not np.any(np.isnan(X)), "NaN found in feature matrix"
assert not np.any(np.isinf(X)), "Inf found in feature matrix"

print("\nPer-feature mean / std (a constant column, std~0, would indicate a bug):")
for i, name in enumerate(FEATURE_NAMES):
    print("  %-16s mean=%.4f  std=%.4f" % (name, X[:, i].mean(), X[:, i].std()))

print("\nSamples per class in the new 23-dim consensus training set (audit 3.4):")
for ci, cname in enumerate(ISIC_CLASSES):
    print("  %-4s %d" % (cname, int((y == ci).sum())))

# --- Stratified split; fit StandardScaler on TRAIN only (no leakage) -------
from sklearn.model_selection import train_test_split
try:
    feat_train_idx, feat_val_idx = train_test_split(
        np.arange(len(y)), test_size=0.20, stratify=y, random_state=42)
except ValueError as exc:
    print("Stratified split failed (%s); falling back to a non-stratified split." % exc)
    feat_train_idx, feat_val_idx = train_test_split(
        np.arange(len(y)), test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X[feat_train_idx])
X_val = scaler.transform(X[feat_val_idx])
y_train = y[feat_train_idx]
y_val = y[feat_val_idx]
print("\nTrain:", X_train.shape, " Val:", X_val.shape,
      "  (scaler fit on the training split only -- no data leakage)")

joblib.dump(scaler, "/kaggle/working/consensus_scaler.pkl")
with open("/kaggle/working/consensus_scaler.json", "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()}, f)
print("Saved scaler -> consensus_scaler.pkl (+ consensus_scaler.json sidecar for the backend)")

# Tensors consumed by the training / temperature-scaling / ECE cells below.
Xtr = torch.tensor(X_train, dtype=torch.float32); ytr = torch.tensor(y_train, dtype=torch.long)
Xva = torch.tensor(X_val, dtype=torch.float32);   yva = torch.tensor(y_val, dtype=torch.long)


## 6. Train the LightGBM consensus head

Replaces the MLP with a LightGBM gradient-boosted decision tree.

**Research basis:**
- ISIC 2024 winning solutions (arxiv 2506.03420) use LightGBM on CNN output features.
- NeurIPS 2024 (Holzmüller et al.) proves well-tuned GBDTs match or beat MLPs on datasets of this size (~3k samples, 23 features).
- LightGBM is invariant to monotonic feature scaling, so the existing `consensus_scaler.pkl` is kept for pipeline compatibility only.

**Cost-sensitive weighting:**
- Base: sqrt-inverse-frequency, clamped to 3× minimum.
- MEL boosted 2.5× (MEL->NV was 20.4% in MLP, target < 15%).
- SCC boosted 2.0× (SCC precision was 0.28 in MLP, target > 0.40).

**Validation:** 5-fold stratified CV for reliable estimates, then final model on train split.

In [ ]:
# ============================================================================
# §6 — LightGBM consensus head: train, validate, calibrate, sweep, save
# ============================================================================

# --- 1.1 Imports ---
import lightgbm as lgbm
from lightgbm import LGBMClassifier
import joblib
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score,
    classification_report, confusion_matrix,
)
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "figures").mkdir(exist_ok=True)

# --- 1.2 Sample weights with cost-sensitive boost ---
# Base weights: sqrt inverse frequency, clamped to 3x minimum
# Research basis: Gayatri & Aarthy 2024 (SAGE) for ISIC-2019 imbalance
class_counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(float)
sqrt_w = 1.0 / np.sqrt(class_counts + 1e-8)
sqrt_w = np.clip(sqrt_w, None, 3.0 * sqrt_w.min())
# Normalize so mean weight = 1 (prevents overall gradient scale shift)
sqrt_w = sqrt_w / sqrt_w.sum() * NUM_CLASSES

sample_weights = np.array([sqrt_w[y] for y in y_train])

# Cost-sensitive boost for clinically dangerous confusions
# Research basis: Springer AI Review 2024 (173-paper survey, CSL for medical)
MEL_IDX  = ISIC_CLASSES.index("MEL")
SCC_IDX  = ISIC_CLASSES.index("SCC")
NV_IDX   = ISIC_CLASSES.index("NV")
BCC_IDX  = ISIC_CLASSES.index("BCC")
AK_IDX   = ISIC_CLASSES.index("AK")

# MEL->NV is the highest-cost error (20.4% in MLP vs 12.5% ensemble)
sample_weights[y_train == MEL_IDX] *= 2.5
# SCC precision was 0.28 in MLP — clearly undertrained
sample_weights[y_train == SCC_IDX] *= 2.0

print("=== Sample weight audit ===")
for cls_idx in range(NUM_CLASSES):
    mask = y_train == cls_idx
    print(f"  {ISIC_CLASSES[cls_idx]:4s}: n={mask.sum():4d}  "
          f"mean_w={sample_weights[mask].mean():.3f}  "
          f"base_sqrt_w={sqrt_w[cls_idx]:.3f}")

# --- 1.3 LightGBM hyperparameters ---
LGBM_PARAMS = dict(
    n_estimators=1000,       # early stopping will find optimal
    learning_rate=0.02,      # low lr + early stopping = good generalization
    num_leaves=31,           # conservative for 3k samples
    max_depth=6,
    min_child_samples=10,    # prevents overfitting on rare classes (DF=33)
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight="balanced", # additional imbalance handling on top of weights
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# --- 1.4 5-fold Stratified Cross-Validation ---
# Required for reliable estimates on ~3k samples.
# Without CV a single lucky val split can show false improvement.
# Reports mean ± std — if std > 0.03 the result is not stable.

print("\n=== 5-Fold Stratified Cross-Validation ===")
print("(Combines train+val for CV, then trains final model on train only)\n")

X_all = np.concatenate([X_train, X_val])
y_all = np.concatenate([y_train, y_val])
w_all = np.array([sqrt_w[y] for y in y_all])
w_all[y_all == MEL_IDX] *= 2.5
w_all[y_all == SCC_IDX] *= 2.0

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_bal_accs, cv_aucs, cv_mel_nv_rates = [], [], []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_all, y_all)):
    X_tr, X_vl = X_all[tr_idx], X_all[vl_idx]
    y_tr, y_vl = y_all[tr_idx], y_all[vl_idx]
    w_tr       = w_all[tr_idx]

    fm = LGBMClassifier(**LGBM_PARAMS)
    fm.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[
            lgbm.early_stopping(50, verbose=False),
            lgbm.log_evaluation(period=0),
        ],
    )
    proba_vl = fm.predict_proba(X_vl)
    pred_vl  = proba_vl.argmax(axis=1)

    bal  = balanced_accuracy_score(y_vl, pred_vl)
    try:
        auc = roc_auc_score(
            y_vl, proba_vl, multi_class="ovr",
            average="macro", labels=list(range(NUM_CLASSES))
        )
    except Exception:
        auc = float("nan")

    # MEL->NV rate per fold
    mel_mask   = (y_vl == MEL_IDX)
    mel_as_nv  = ((pred_vl == NV_IDX) & mel_mask).sum()
    mel_nv_rate_fold = mel_as_nv / max(mel_mask.sum(), 1)

    cv_bal_accs.append(bal)
    cv_aucs.append(auc)
    cv_mel_nv_rates.append(mel_nv_rate_fold)
    print(f"  Fold {fold+1}: bal_acc={bal:.4f}  "
          f"macro_auc={auc:.4f}  "
          f"MEL->NV={mel_as_nv}/{mel_mask.sum()} ({mel_nv_rate_fold:.1%})")

print(f"\nCV balanced accuracy : "
      f"{np.mean(cv_bal_accs):.4f} ± {np.std(cv_bal_accs):.4f}")
print(f"CV macro AUC         : "
      f"{np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}")
print(f"CV MEL->NV rate      : "
      f"{np.mean(cv_mel_nv_rates):.1%} ± {np.std(cv_mel_nv_rates):.1%}")

if np.std(cv_bal_accs) > 0.03:
    print("WARNING: high CV variance (std > 0.03) — result may not be stable")
else:
    print("OK: CV variance is acceptable")

# --- 1.5 Final model on train split ---
print("\n=== Final model (train split -> held-out val split) ===")

lgbm_final = LGBMClassifier(**LGBM_PARAMS)
lgbm_final.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgbm.early_stopping(50, verbose=False),
        lgbm.log_evaluation(period=100),
    ],
)

val_proba   = lgbm_final.predict_proba(X_val)
val_pred    = val_proba.argmax(axis=1)
val_bal_acc = balanced_accuracy_score(y_val, val_pred)

print(f"\nFinal val balanced accuracy: {val_bal_acc:.4f}")
print(f"(MLP was 0.7184 on val, 0.675 on full test split)\n")
print(classification_report(
    y_val, val_pred, target_names=ISIC_CLASSES, zero_division=0
))

# --- 1.6 Primary failure mode report ---
print("=== Primary failure mode audit ===")
cm_val = confusion_matrix(y_val, val_pred, labels=list(range(NUM_CLASSES)))

# MEL->NV (most dangerous confusion)
mel_total    = cm_val[MEL_IDX].sum()
mel_as_nv    = cm_val[MEL_IDX, NV_IDX]
mel_nv_rate  = mel_as_nv / max(mel_total, 1)

# SCC precision and recall
scc_tp       = cm_val[SCC_IDX, SCC_IDX]
scc_pred_tot = cm_val[:, SCC_IDX].sum()
scc_true_tot = cm_val[SCC_IDX].sum()
scc_prec     = scc_tp / max(scc_pred_tot, 1)
scc_rec      = scc_tp / max(scc_true_tot, 1)

print(f"MEL->NV : {mel_as_nv}/{mel_total} = {mel_nv_rate:.1%}"
      f"  target <15%  {'PASS' if mel_nv_rate < 0.15 else 'FAIL'}")
print(f"NV->MEL : {cm_val[NV_IDX, MEL_IDX]}/{cm_val[NV_IDX].sum()}"
      f" = {cm_val[NV_IDX, MEL_IDX]/max(cm_val[NV_IDX].sum(),1):.1%}")
print(f"SCC prec: {scc_prec:.3f}  "
      f"target >0.40  {'PASS' if scc_prec > 0.40 else 'FAIL'}")
print(f"SCC rec : {scc_rec:.3f}")

# --- 1.7 Feature importance + shortcut detection ---
print("\n=== Feature importance (gain) ===")
importance_df = pd.Series(
    lgbm_final.feature_importances_,
    index=FEATURE_NAMES
).sort_values(ascending=False)

print(importance_df.head(10).to_string())

top_share = importance_df.iloc[0] / importance_df.sum()
print(f"\nTop feature share of total: {top_share:.1%}")

# Shortcut detection: single-feature ablation on val set
print("\n=== Single-feature ablation (shortcut check) ===")
top_feat_name = importance_df.index[0]
top_feat_idx  = list(FEATURE_NAMES).index(top_feat_name)

X_ablated = X_val.copy()
X_ablated[:, top_feat_idx] = 0.0
ablated_bal = balanced_accuracy_score(
    y_val, lgbm_final.predict(X_ablated)
)
drop = val_bal_acc - ablated_bal
print(f"Top feature: {top_feat_name}")
print(f"Val bal_acc with feature zeroed: {ablated_bal:.4f}"
      f"  (drop = {drop:.4f})")

if drop > 0.20:
    print("WARNING: removing one feature drops accuracy >0.20"
          " — shortcut still present")
    print("  Consider training without attention feature group (dims 20-22)")
else:
    print("OK: no single-feature collapse — model is using features broadly")

# Attention feature ranks
print("\nAttention feature ranks:")
for f in ["attn_iou", "attn_entropy_a", "attn_entropy_b"]:
    rank  = list(importance_df.index).index(f) + 1
    share = importance_df[f] / importance_df.sum()
    print(f"  {f}: rank {rank}/{len(FEATURE_NAMES)} ({share:.1%} of total)")

# --- 1.8 Calibration — ECE first, apply only if needed ---
# IMPORTANT: calibration is fit using 5-fold CV on training data ONLY.
# The val set is NEVER used for calibration fitting — only for evaluation.
# This avoids the data leakage of fitting calibration on the same val set
# used for model selection.

def compute_ece(probs, labels, n_bins=15):
    """Expected calibration error (binned)."""
    conf    = probs.max(axis=1)
    pred    = probs.argmax(axis=1)
    correct = (pred == labels).astype(float)
    bins    = np.linspace(0, 1, n_bins + 1)
    ece     = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = ((conf >= lo) & (conf < hi)) if i < n_bins - 1 \
            else ((conf >= lo) & (conf <= hi))
        if m.sum() == 0:
            continue
        ece += (m.sum() / len(labels)) * abs(
            correct[m].mean() - conf[m].mean()
        )
    return float(ece)

ece_raw = compute_ece(val_proba, y_val)
print(f"\n=== Calibration audit ===")
print(f"ECE raw LightGBM : {ece_raw:.4f}")
print(f"ECE reference    : MLP=0.054  Ensemble=0.163")

APPLY_CALIBRATION = ece_raw > 0.10
print(f"Apply calibration: {APPLY_CALIBRATION}  (threshold ECE > 0.10)")

if APPLY_CALIBRATION:
    # CalibratedClassifierCV with cv=5 trains calibration on training
    # data folds only — no val leakage.
    print("Fitting isotonic calibration via 5-fold CV on training data...")
    calibrated = CalibratedClassifierCV(
        estimator=LGBMClassifier(**LGBM_PARAMS),
        method="isotonic",
        cv=5,
    )
    calibrated.fit(X_train, y_train, sample_weight=sample_weights)

    val_proba_cal = calibrated.predict_proba(X_val)
    ece_cal = compute_ece(val_proba_cal, y_val)
    print(f"ECE after calibration: {ece_cal:.4f}")

    if ece_cal < ece_raw:
        final_model = calibrated
        final_proba = val_proba_cal
        print("Calibration helped — using calibrated model.")
    else:
        final_model = lgbm_final
        final_proba = val_proba
        print("Calibration did not help — using raw LightGBM.")
else:
    final_model = lgbm_final
    final_proba = val_proba
    print("ECE acceptable — no calibration applied.")

# Reliability diagram
fig, ax = plt.subplots(figsize=(6, 5))
conf_all    = final_proba.max(axis=1)
pred_all    = final_proba.argmax(axis=1)
correct_all = (pred_all == y_val).astype(float)
bins_plot   = np.linspace(0, 1, 11)
bc, ba = [], []
for i in range(len(bins_plot) - 1):
    m = (conf_all >= bins_plot[i]) & (conf_all < bins_plot[i + 1])
    if m.sum() > 0:
        bc.append((bins_plot[i] + bins_plot[i + 1]) / 2)
        ba.append(correct_all[m].mean())
ax.plot(bc, ba, "b-o", label="LightGBM", linewidth=2)
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
ax.set_title(f"Reliability diagram  ECE={compute_ece(final_proba, y_val):.4f}")
ax.legend(); fig.tight_layout()
fig.savefig(WORK_DIR / "figures" / "lgbm_reliability.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/lgbm_reliability.png")

# --- 1.9 Trigger threshold sweep ---
# IMPLEMENTATION NOTE:
# In the real Argus pipeline, non-triggered samples have dims 20-22 = 0.0
# because Grad-CAM and attention rollout are skipped.
# Dims 0-19 do NOT depend on the trigger decision.
# CAVEAT: X_val was standardized by consensus_scaler. The correct simulation
# requires working in RAW feature space.

print("\n=== Trigger threshold sweep ===")
print("Simulating full Argus decision for each (TAU_JS, TAU_ENTROPY) pair\n")

# Invert scaler to get raw features
# consensus_scaler is a StandardScaler: raw = scaled * scale_ + mean_
try:
    scale_ = scaler.scale_
    mean_  = scaler.mean_
except AttributeError:
    scale_ = np.asarray(scaler.scale_)
    mean_  = np.asarray(scaler.mean_)
X_val_raw = X_val * scale_ + mean_

TAU_JS_OPTIONS      = [0.15, 0.20, 0.25, 0.30]
TAU_ENTROPY_OPTIONS = [0.60, 0.70, 0.80, 0.90]

sweep_rows = []
for tjs in TAU_JS_OPTIONS:
    for tent in TAU_ENTROPY_OPTIONS:
        js_vals    = X_val_raw[:, 16]
        ent_a_vals = X_val_raw[:, 17]
        ent_b_vals = X_val_raw[:, 18]

        # Exact trigger simulation
        would_fire = (js_vals > tjs) | (
            np.maximum(ent_a_vals, ent_b_vals) > tent
        )

        # Build raw feature matrix: zero attn dims for non-triggered
        X_sim_raw = X_val_raw.copy()
        X_sim_raw[~would_fire, 20:23] = 0.0

        # Re-scale for LightGBM (pipeline compatibility)
        X_sim_scaled = (X_sim_raw - mean_) / np.where(scale_ == 0, 1, scale_)

        sim_proba = final_model.predict_proba(X_sim_scaled)
        sim_pred  = sim_proba.argmax(axis=1)
        bal       = balanced_accuracy_score(y_val, sim_pred)

        mel_as_nv_sim  = ((y_val == MEL_IDX) & (sim_pred == NV_IDX)).sum()
        mel_total_sim  = (y_val == MEL_IDX).sum()
        mel_nv_sim     = mel_as_nv_sim / max(mel_total_sim, 1)

        try:
            auc = roc_auc_score(
                y_val, sim_proba, multi_class="ovr",
                average="macro", labels=list(range(NUM_CLASSES))
            )
        except Exception:
            auc = float("nan")

        sweep_rows.append({
            "TAU_JS"      : tjs,
            "TAU_ENTROPY" : tent,
            "fire_rate"   : round(float(would_fire.mean()), 3),
            "val_bal_acc" : round(bal, 4),
            "macro_auc"   : round(auc, 4),
            "MEL_NV_rate" : round(mel_nv_sim, 4),
        })

sweep_df = pd.DataFrame(sweep_rows).sort_values(
    "val_bal_acc", ascending=False
)
print(sweep_df.to_string(index=False))

# Best: highest bal_acc where MEL->NV < 15%
valid = sweep_df[sweep_df["MEL_NV_rate"] < 0.15]
if len(valid) > 0:
    best = valid.iloc[0]
    print(f"\nBest threshold (bal_acc maximized AND MEL->NV < 15%):")
else:
    best = sweep_df.iloc[0]
    print(f"\nNo threshold achieves MEL->NV < 15%. Best by bal_acc:")

print(f"  TAU_JS={best['TAU_JS']}  TAU_ENTROPY={best['TAU_ENTROPY']}")
print(f"  val_bal_acc={best['val_bal_acc']}  "
      f"fire_rate={best['fire_rate']}  "
      f"MEL->NV={best['MEL_NV_rate']:.1%}")
print(f"\nACTION: Update TAU_JS and TAU_ENTROPY constants in BOTH"
      f" NB04 and NB05 to these values if they differ from current 0.25/0.80")

# --- 1.10 Save all outputs ---
print("\n=== Saving outputs ===")

# Primary checkpoint
joblib.dump(final_model, WORK_DIR / "consensus_lgbm.pkl")
print("Saved: consensus_lgbm.pkl")

# JSON sidecar (read by NB05 on load)
sidecar = {
    "model_type"              : "lightgbm",
    "calibrated"              : APPLY_CALIBRATION,
    "n_classes"               : NUM_CLASSES,
    "classes"                 : ISIC_CLASSES,
    "feature_names"           : list(FEATURE_NAMES),
    "lgbm_params"             : {k: v for k, v in LGBM_PARAMS.items() if not callable(v)},
    "val_balanced_accuracy"   : float(val_bal_acc),
    "val_ece_raw"             : float(ece_raw),
    "val_mel_nv_rate"         : float(mel_nv_rate),
    "val_scc_precision"       : float(scc_prec),
    "cv_bal_acc_mean"         : float(np.mean(cv_bal_accs)),
    "cv_bal_acc_std"          : float(np.std(cv_bal_accs)),
    "cv_macro_auc_mean"       : float(np.mean(cv_aucs)),
    "cv_macro_auc_std"        : float(np.std(cv_aucs)),
    "cv_mel_nv_mean"          : float(np.mean(cv_mel_nv_rates)),
    "best_trigger_tau_js"     : float(best["TAU_JS"]),
    "best_trigger_tau_entropy": float(best["TAU_ENTROPY"]),
}
with open(WORK_DIR / "consensus_lgbm.json", "w") as f:
    json.dump(sidecar, f, indent=2)
print("Saved: consensus_lgbm.json")

# Feature importance CSV
importance_df.to_csv(WORK_DIR / "lgbm_feature_importance.csv",
                     header=["importance"])
print("Saved: lgbm_feature_importance.csv")

# Confusion matrix plot
fig2, ax2 = plt.subplots(figsize=(7, 6))
import seaborn as sns
sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues",
            xticklabels=ISIC_CLASSES, yticklabels=ISIC_CLASSES, ax=ax2)
ax2.set_title("LightGBM consensus confusion matrix (val)")
ax2.set_xlabel("Predicted"); ax2.set_ylabel("True")
fig2.tight_layout()
fig2.savefig(WORK_DIR / "figures" / "consensus_confusion_matrix.png",
             dpi=120, bbox_inches="tight")
plt.show()
print("Saved: figures/consensus_confusion_matrix.png")

# Sanity check: reload the saved model
_check = joblib.load(WORK_DIR / "consensus_lgbm.pkl")
_test_proba = _check.predict_proba(X_val[:2])
assert _test_proba.shape == (2, NUM_CLASSES), \
    f"Reload check failed: expected shape (2,{NUM_CLASSES}), got {_test_proba.shape}"
print("Reload sanity check passed.")

# Summary printout
print("\n=== TRAINING COMPLETE — SUMMARY ===")
print(f"CV  bal_acc  : {np.mean(cv_bal_accs):.4f} +/- {np.std(cv_bal_accs):.4f}")
print(f"Val bal_acc  : {val_bal_acc:.4f}  (MLP was 0.7184 on val)")
print(f"Val ECE      : {ece_raw:.4f}  (MLP was 0.054)")
print(f"MEL->NV      : {mel_nv_rate:.1%}  target <15%"
      f"  {'PASS' if mel_nv_rate < 0.15 else 'FAIL'}")
print(f"SCC precision: {scc_prec:.3f}  target >0.40"
      f"  {'PASS' if scc_prec > 0.40 else 'FAIL'}")
print(f"Calibrated   : {APPLY_CALIBRATION}")
print(f"Best trigger : TAU_JS={best['TAU_JS']}"
      f"  TAU_ENTROPY={best['TAU_ENTROPY']}")
print(f"\nNext: upload consensus_lgbm.pkl + consensus_lgbm.json to"
      f" argus-checkpoints-final dataset on Kaggle, then run NB05.")

## 7. Outputs and how to use them

All artifacts are written to **`/kaggle/working/`**:

| File | Purpose |
|------|---------|
| `consensus_lgbm.pkl` | Trained LightGBM consensus head (possibly calibrated). |
| `consensus_lgbm.json` | Sidecar with metrics, params, and best trigger thresholds. |
| `consensus_scaler.pkl` | StandardScaler fit on the consensus training split (kept for pipeline compatibility). |
| `consensus_scaler.json` | `{"mean", "scale"}` sidecar so the backend needs only numpy. |
| `lgbm_feature_importance.csv` | Feature importance (gain) for the LightGBM model. |
| `figures/*.png` | Reliability diagram, confusion matrix. |

### Hand off to NB05 (evaluation)
Save a version so `/kaggle/working` is published, then in NB05 *Add Data → Notebook Output* and attach it. `find_file('consensus_lgbm')` locates the checkpoint. The scaler **must** also be attached.

## 10. Outputs and how to use them

All artifacts are written to **`/kaggle/working/`**:

| File | Purpose |
|------|---------|
| `consensus_best.pth` | Trained 23-dim consensus MLP `state_dict` (incl. `temperature`) + `ece`. |
| `consensus_scaler.pkl` | StandardScaler fit on the consensus training split. **Required** by NB05 and the backend. |
| `consensus_scaler.json` | `{"mean", "scale"}` sidecar so the backend needs only numpy. |
| `consensus_temperature.txt` | The scalar calibration temperature. |
| `figures/*.png` | Training curves, confusion matrix, reliability diagram. |

### Hand off to NB05 (evaluation)
Save a version so `/kaggle/working` is published, then in NB05 *Add Data -> Notebook Output* and attach it. `find_file('consensus_best')` and `find_file('consensus_scaler')` locate the checkpoint and scaler. The scaler **must** be attached or NB05/the backend will apply an identity transform and the predictions will be miscalibrated.

> The Groq response cache and the `.npy` feature caches are gone: feature building is now cheap enough (no LLM calls) to run every time.